In [0]:
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingestor import read_jdbc, write_raw
import json


# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("sql-server-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]
write_options = config["write_options"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat()
logger.info("Load date is %s",run_date )
logger.info("Reading data from  %s",source_config["dbtable"] )




user = dbutils.secrets.get(scope="retail-platform-devv", key = source_config["user"])
password = dbutils.secrets.get(scope="retail-platform-devv", key = source_config["password"])

# ============================================================
# 5. READ FROM SQL SERVER
# ============================================================
df = read_jdbc(spark, source_config["url"],source_config["dbtable"], user, password  )
logger.info("read %s rows", df.count())
logger.info("sample data.......")
df.show()


# ============================================================
# 6. TARGET PATH
# ============================================================

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

# ============================================================
# 7. WRITE RAW
# ============================================================
write_raw(df, target_path,target_config["file_format"], target_config["mode"], write_options)
logger.info("data landed at %s", target_path)


